<a href="https://colab.research.google.com/github/1900690/depth-estimation/blob/main/depth_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Depth pro で深度推定

In [1]:
import os

# --- 環境チェックと自動セットアップ ---
try:
    # すでに環境が整っているか確認
    import depth_pro
    import numpy as np
    # NumPyのバイナリ互換性エラーがないかテスト
    np.zeros(1).dtype
except (ImportError, ValueError):
    print("環境の初期設定が必要です。インストールを開始し、完了後にランタイムを再起動します...")

    # 1. ライブラリのインストールとモデルの取得（ログは最小限に抑制）
    os.system("pip install -q --upgrade numpy")
    os.system("pip install -q git+https://github.com/apple/ml-depth-pro.git")

    os.makedirs("checkpoints", exist_ok=True)
    os.system('wget -q -L "https://huggingface.co/apple/DepthPro/resolve/main/depth_pro.pt?download=true" -O checkpoints/depth_pro.pt')

    # 2. ランタイムを強制終了して再起動をトリガー
    # Colabは終了を検知すると自動で新しいセッションを準備します
    print("!!! ランタイムを再起動します。このセルを再実行してください !!!")
    os._exit(0)

# --- 再起動後、または環境構築済みの場合はここから実行 ---
import torch
import warnings
import logging

# 不要な警告メッセージを非表示にする
warnings.filterwarnings("ignore")
logging.getLogger("torch").setLevel(logging.ERROR)

# 3. モデルのロード
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, transform = depth_pro.create_model_and_transforms()
model = model.to(device).eval()

print(f"セットアップ完了: {device} 上で Depth Pro がロードされました。")

セットアップ完了: cuda 上で Depth Pro がロードされました。


In [19]:
#@title URLで入手
import os

# --- 設定 ---
# ダウンロードしたい動画の直リンクURLを入力してください
file_url = "https://github.com/1900690/image-movie-editing/releases/download/avi_isida2/20251015.AVI"
# 保存するファイル名
target_name = "input.mp4"

# --- 実行 ---
# 既存のファイルを削除して上書きできるようにする
if os.path.exists(target_name):
    os.remove(target_name)

print(f"ファイルをダウンロード中: {file_url}")

# -O オプションで保存名を指定してダウンロード
!wget -L "{file_url}" -O {target_name}

if os.path.exists(target_name):
    print(f"✅ '{target_name}' として保存が完了しました。")
else:
    print("❌ ダウンロードに失敗しました。URLが正しいか確認してください。")

ファイルをダウンロード中: https://github.com/1900690/image-movie-editing/releases/download/avi_isida2/20251015.AVI
--2026-01-26 06:03:39--  https://github.com/1900690/image-movie-editing/releases/download/avi_isida2/20251015.AVI
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/925939876/7691e994-3a1c-4a14-a661-158e32dcc6ee?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-26T06%3A49%3A36Z&rscd=attachment%3B+filename%3D20251015.AVI&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-26T05%3A48%3A48Z&ske=2026-01-26T06%3A49%3A36Z&sks=b&skv=2018-11-09&sig=jF7tdzQ5O6XdKkP1aa%2Fm5PyyVphtSif3mspgxagMHRQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29

In [14]:
import os
from google.colab import files

# 動画アップロード
print("動画ファイルをアップロードしてください")
uploaded = files.upload()

if uploaded:
    # アップロードされた最初のファイル名を取得
    original_name = list(uploaded.keys())[0]

    # 既存の input.mp4 があれば削除（エラー回避用）
    if os.path.exists("input.mp4"):
        os.remove("input.mp4")

    # ファイルを input.mp4 にリネーム
    os.rename(original_name, "input.mp4")

    print(f"'{original_name}' を 'input.mp4' として保存しました。")

動画ファイルをアップロードしてください


In [ ]:
#@title 🛠 深度解析・強調フォーム { display-mode: "form" }

#@markdown ### 📂 ファイルパス設定
input_path = "input.mp4" #@param {type:"string"}
output_path = "output_optimized.mp4" #@param {type:"string"}

#@markdown ### ⚙️ 基本動作設定
frame_interval = 5 #@param {type:"slider", min:1, max:30, step:1}
mask_height_px = 70 #@param {type:"number"}
#@markdown <small>※画面下部を切り取る高さ(px)</small>

#@markdown ### 🎯 深度強調（レンジ指定）設定
#@markdown <small>16m地点の微小な差を見たい場合、Target=16, Width=2 などに設定します</small>
target_depth = 16.0 #@param {type:"number"}
range_width = 2.0 #@param {type:"number"}
use_dynamic_range = False #@param {type:"boolean"}
#@markdown <small>※Trueにすると上記設定を無視して各フレームの最小・最大に合わせます</small>

#@markdown ### 📏 距離補正（キャリブレーション）
use_ref_correction = False #@param {type:"boolean"}
ref_point_x = 640 #@param {type:"number"}
ref_point_y = 400 #@param {type:"number"}
ref_distance = 16.0 #@param {type:"number"}

# --- 設定を辞書に集約 ---
CONFIG = {
    "input_path": input_path,
    "output_path": output_path,
    "frame_interval": frame_interval,
    "mask_height_px": mask_height_px,
    "target_depth": target_depth,
    "range_width": range_width,
    "use_dynamic_range": use_dynamic_range,
    "use_ref_correction": use_ref_correction,
    "ref_point": (ref_point_x, ref_point_y),
    "ref_distance": ref_distance
}

import torch
import depth_pro
import numpy as np
import cv2
import PIL.Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

# --- カメラパラメータ ---
DIM = (1280, 720)
K = np.array([[768.4324165833522, 0.0, 629.193977277274],
              [0.0, 768.0867082519276, 363.13481714845415],
              [0.0, 0.0, 1.0]])
D = np.array([[-0.07176022472676662], [-0.01831839322401357],
              [-0.051792564889714565], [0.0897998816199078]])

new_K = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(K, D, DIM, np.eye(3), balance=0)
map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), new_K, DIM, cv2.CV_16SC2)

def robust_vmin_vmax(depth_m, cfg):
    """    depth_m: HxW float (NaN/inf/0 を含む可能性あり)
    戻り値: (vmin, vmax, mode_string)
    """
    d = depth_m.astype(np.float32)

    # 無効値除去（Depth推定で 0 や inf が混じるケース対策）
    valid = np.isfinite(d) & (d > 0)
    if valid.sum() < 100:
        return 0.0, 1.0, "fallback(no_valid)"

    dv = d[valid]

    # パーセンタイルで頑健にレンジを作る（黒潰れ防止の要）
    p1, p99 = np.percentile(dv, [1, 99])
    if (not np.isfinite(p1)) or (not np.isfinite(p99)) or (p99 - p1) < 1e-6:
        mn, mx = float(dv.min()), float(dv.max())
        if mx - mn < 1e-6:
            return mn - 0.5, mn + 0.5, "fallback(const)"
        return mn, mx, "fallback(minmax)"

    if cfg["use_dynamic_range"]:
        return float(p1), float(p99), "dynamic(p1-p99)"

    vmin_t = cfg["target_depth"] - (cfg["range_width"] / 2.0)
    vmax_t = cfg["target_depth"] + (cfg["range_width"] / 2.0)

    inter = max(0.0, min(vmax_t, p99) - max(vmin_t, p1))
    if inter <= 0.01 * (p99 - p1):
        return float(p1), float(p99), "fallback_to_dynamic(p1-p99)"

    return float(vmin_t), float(vmax_t), "target"


def create_depth_visualization_robust(depth, width, height, cfg):
    vmin, vmax, mode = robust_vmin_vmax(depth, cfg)

    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
    im = ax.imshow(depth, cmap='turbo', vmin=vmin, vmax=vmax)

    ax.set_title(f"{mode}: {vmin:.2f} - {vmax:.2f}", fontsize=15)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Depth (model units)', rotation=270, labelpad=15)

    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    img_viz = cv2.cvtColor(np.asarray(canvas.buffer_rgba()), cv2.COLOR_RGBA2BGR)
    plt.close(fig)
    return cv2.resize(img_viz, (width, height))


def create_depth_visualization(depth, width, height, cfg):
    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)

    if cfg["use_dynamic_range"]:
        v_min, v_max = depth.min(), depth.max()
    else:
        # 指定された範囲に色を集中させる
        v_min = cfg["target_depth"] - (cfg["range_width"] / 2)
        v_max = cfg["target_depth"] + (cfg["range_width"] / 2)

    # cmap='turbo' は 'rainbow' より微細な変化のコントラストがつきやすいです
    im = ax.imshow(depth, cmap='turbo', vmin=v_min, vmax=v_max)

    ax.set_title(f"Focus: {v_min:.1f}m - {v_max:.1f}m", fontsize=15)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Depth [m]', rotation=270, labelpad=15)

    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    img_viz = cv2.cvtColor(np.asarray(canvas.buffer_rgba()), cv2.COLOR_RGBA2BGR)
    plt.close(fig)
    return cv2.resize(img_viz, (width, height))

def run_depth_estimation_system(cfg):
    cap = cv2.VideoCapture(cfg["input_path"])
    if not cap.isOpened():
        print(f"エラー: {cfg['input_path']} が見つかりません。")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_h = DIM[1] - cfg["mask_height_px"]
    out_w = DIM[0]

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(cfg["output_path"], fourcc, fps, (out_w * 2, out_h))

    pbar = tqdm(total=total_frames, desc="解析中")
    frame_idx = 0
    middle_frame_img = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % cfg["frame_interval"] == 0:
            # 1. 魚眼補正
            undistorted = cv2.remap(frame, map1, map2, interpolation=cv2.INTER_LINEAR)

            # 2. 推論（入力にはマスクしない：推論を壊さないため）
            img_inf = cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB)

            input_data = transform(PIL.Image.fromarray(img_inf))
            if torch.cuda.is_available(): input_data = input_data.cuda()

            with torch.no_grad():
                prediction = model.infer(input_data)
                depth = prediction["depth"].cpu().numpy()

            # --- デバッグ: 1フレーム目の深度統計を表示（黒潰れ原因の切り分け） ---
            if frame_idx == 0:
                dv = depth[np.isfinite(depth) & (depth > 0)]
                if dv.size:
                    print("depth stats:",
                          "min", float(dv.min()),
                          "p1",  float(np.percentile(dv,1)),
                          "p50", float(np.percentile(dv,50)),
                          "p99", float(np.percentile(dv,99)),
                          "max", float(dv.max()))
                else:
                    print("depth stats: no valid depth pixels")


            # 3. 距離補正 (オプション)
            if cfg["use_ref_correction"]:
                px_x, px_y = cfg["ref_point"]
                predicted_val = depth[px_y, px_x]
                if predicted_val > 0:
                    depth *= (cfg["ref_distance"] / predicted_val)

            # 4. クロップ処理（出力側だけ下部をカット：推論入力はそのまま）
            undistorted_c = undistorted[:out_h, :]
            depth_c = depth[:out_h, :]

            # 5. 可視化 (強調レンジ適用)
            depth_viz = create_depth_visualization_robust(depth_c, out_w, out_h, cfg)

            # 6. 結合
            combined = np.hstack((undistorted_c, depth_viz))

            # 補正点の確認用マーカー
            if cfg["use_ref_correction"] and cfg["ref_point"][1] < out_h:
                cv2.circle(combined, cfg["ref_point"], 7, (0, 255, 0), -1)

            out.write(combined)

            if frame_idx >= (total_frames // 2) and middle_frame_img is None:
                middle_frame_img = combined.copy()

        frame_idx += 1
        pbar.update(1)

    cap.release()
    out.release()
    print(f"\n✅ 完了: {cfg['output_path']}")

    if middle_frame_img is not None:
        plt.figure(figsize=(15, 8))
        plt.imshow(cv2.cvtColor(middle_frame_img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("強調表示の確認（中間フレーム）")
        plt.show()

# 実行
run_depth_estimation_system(CONFIG)

In [24]:
from google.colab import files
import os

# 解析結果のファイル名（前のステップで指定したもの）
output_filename = "output_optimized.mp4"

if os.path.exists(output_filename):
    print(f"📥 '{output_filename}' をダウンロードします...")
    files.download(output_filename)
else:
    print(f"❌ エラー: '{output_filename}' が見つかりません。解析が完了しているか確認してください。")

📥 'output_optimized.mp4' をダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>